<a href="https://colab.research.google.com/github/ishach20-a11y/Master-thesis/blob/code/LLama_Groq_zero_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# experiment_runner_groq_llama_zero_shot.py

# ── 0. Setup ───────────────────────────────────────────────
!pip install -q groq

from groq import Groq
from google.colab import files
from google.colab import userdata
import os
import re

os.makedirs("experiments/dmn", exist_ok=True)

In [ ]:
# ── 1. API key and model ───────────────────────────────────
API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=API_KEY)

MODEL_NAME = "llama-3.1-8b-instant"
# Alternative:
# MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"

# ── 2. Configuration ───────────────────────────────────────
# Change manually per run: 0.2 / 0.4 / 0.6
TEMPERATURE = 0.2

N_ITERATIONS = 3
MAX_TOKENS = 30000
TOP_P = 1.0
FREQUENCY_PENALTY = 0.0
PRESENCE_PENALTY = 0.6

# ── 3. New description ─────────────────────────────────────
description_id = "description_1"

description = """PASTE NEW DESCRIPTION HERE"""

# ── 4. Zero-shot prompt ────────────────────────────────────
def build_zero_shot_prompt(description: str) -> str:
    return f"""<s>[INST] You are an expert in DMN diagram generation.

Generate a complete Camunda-compatible DMN XML file for the new DMN textual description provided below.

Important requirements:
- Return only complete DMN XML.
- Do not repeat the input description.
- Do not include explanations, headings, labels, or Markdown.
- The output must start with <?xml version="1.0" encoding="UTF-8"?>
- The output must include <definitions>.
- The output must include valid DMN namespace declarations.
- The output must include decisions, inputData elements, informationRequirement elements, and DMNDI diagram elements.
- The file must be importable and readable in Camunda Modeler.
- Do not produce a simplified XML fragment.
- Do not stop before closing </definitions>.

Text: '{description}' [/INST]</s>"""

# ── 5. Clean model output ──────────────────────────────────
def clean_model_output(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```xml\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    xml_start = text.find("<?xml")
    if xml_start == -1:
        xml_start = text.find("<definitions")

    if xml_start != -1:
        text = text[xml_start:]

    xml_end = text.rfind("</definitions>")
    if xml_end != -1:
        text = text[:xml_end + len("</definitions>")]

    return text.strip()

# ── 6. Run generation ──────────────────────────────────────
prompt = build_zero_shot_prompt(description)

for iteration in range(1, N_ITERATIONS + 1):
    print(f"▶ Groq LLaMA zero-shot | {description_id} | temp={TEMPERATURE} | iter={iteration}")

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=TEMPERATURE,
        max_completion_tokens=MAX_TOKENS,
        top_p=TOP_P,
        frequency_penalty=FREQUENCY_PENALTY,
        presence_penalty=PRESENCE_PENALTY,
    )

usage = response.usage_metadata

    print("Input tokens:", usage.prompt_token_count)
    print("Output tokens:", usage.candidates_token_count)
    print("Total tokens:", usage.total_token_count)

    dmn_xml = clean_model_output(response.choices[0].message.content)

    base_name = f"{description_id}_groq_llama_zero_shot_temp_{TEMPERATURE}_iter_{iteration}"
    dmn_path = f"experiments/dmn/{base_name}.dmn"

    with open(dmn_path, "w", encoding="utf-8") as f:
        f.write(dmn_xml)

    print(f"Saved: {dmn_path}")

# ── 7. Download DMN results ────────────────────────────────
for iteration in range(1, N_ITERATIONS + 1):
    base_name = f"{description_id}_groq_llama_zero_shot_temp_{TEMPERATURE}_iter_{iteration}"
    files.download(f"experiments/dmn/{base_name}.dmn")